# Vorhersage von Kreditausfällen

Ein Modell für die Ausfallwahrscheinlichkeit (Probability of Default, PD) auf dem
LendingClub-Datensatz (2007–2018Q4). Das Notebook enthält die explorative
Analyse, die Grafiken und die Erläuterungen; Datenaufbereitung, Training und
Evaluation liegen in den importierbaren Modulen unter `src/` (siehe Issue #4).

Dies ist die deutsche Fassung von `prediction.ipynb`. Code und Ablauf sind
identisch; übersetzt sind Texte, Kommentare und Ausgaben.

**Ausführen:** `requirements.txt` installieren, die Rohdatei
`accepted_2007_to_2018Q4.csv` in `02_data/raw/` ablegen und das Notebook von oben
nach unten ausführen („Restart & Run All“). Ohne Oberfläche geht das mit:

```
jupyter nbconvert --to notebook --execute --inplace 01_notebooks/prediction_de.ipynb
```

Die Roh-CSV (~1,6 GB) ist nicht Teil dieses Repositorys; sie stammt aus dem
LendingClub-Datensatz auf Kaggle.

**Abgrenzung.** Das Modell schätzt ausschließlich eine Ausfallwahrscheinlichkeit.
LGD (Loss Given Default, Verlustquote bei Ausfall) und EAD (Exposure at Default,
Forderungshöhe bei Ausfall), die beiden anderen Größen, die eine Bank für einen
erwarteten Verlust oder eine Eigenkapitalanforderung braucht, werden hier nicht
modelliert. Zudem ist dies ein Portfolio-Projekt auf öffentlichen historischen
Daten und kein aufsichtlich genehmigtes Ratingsystem: Das Zielmerkmal ist eine
Abschreibung statt der regulatorischen Ausfalldefinition, der Horizont ist die
gesamte Kreditlaufzeit statt eines Jahres, und die Kalibrierung und Validierung,
die ein Ratingsystem erfordert, wurden nicht durchgeführt. Der README-Abschnitt
„Regulatory context“ ordnet das Projekt gegenüber Basel und IFRS 9 ein und
beschreibt, was bewusst ausgeklammert bleibt.

In [ ]:
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
from sklearn.calibration import CalibrationDisplay

# Das Notebook liegt in 01_notebooks/, das Paket src im Projektstamm.
sys.path.insert(0, "..")

from src.data_processing import (
    add_issue_year,
    create_target,
    drop_leakage_columns,
    filter_issue_years,
    load_raw_data,
    rm_nas,
)
from src.evaluate import evaluate_model
from src.features import engineer_features
from src.train import (
    save_model,
    split_and_scale,
    train_random_forest,
    train_scorecard,
)

sns.set_style("whitegrid")

## 1. Daten laden

Die Rohdatei umfasst rund 2,2 Mio. Zeilen; Laden und Aufbereiten dauern daher
einige Minuten und benötigen viel Arbeitsspeicher. Für einen schnellen Durchlauf
durch das Notebook kann `load_raw_data` mit `nrows` aufgerufen werden, um nur
eine Stichprobe zu lesen.

In [ ]:
print("Lade den vollständigen Datensatz... Das kann einige Minuten dauern.")
df = load_raw_data("../02_data/raw/accepted_2007_to_2018Q4.csv")

# Für schnelle Tests mit einer kleineren Stichprobe:
# df = load_raw_data("../02_data/raw/accepted_2007_to_2018Q4.csv", nrows=100000)

print(f"Dimensionen des Datensatzes: {df.shape}")

## 2. Explorative Datenanalyse

Ein erster Blick auf Umfang, Verteilungen und Korrelationsstruktur der Daten.
Die Grafik nach Jahren begründet die einzige Einschränkung, die bereits in
diesem Abschnitt vorgenommen wird: den Filter auf das Zeitfenster 2015–2018
einige Zellen weiter unten.

In [ ]:
df.info()

In [ ]:
df.shape

In [ ]:
df.describe()

In [ ]:
df['grade'].value_counts()

In [ ]:
# Die Kreditbeträge sind das ausfallgefährdete Exposure; ihre Verteilung gibt
# daher den Maßstab für alles Folgende vor.
plt.figure(figsize=(10, 6))
sns.histplot(df['loan_amnt'], bins=50, kde=True)
plt.title('Verteilung der Kreditbeträge')
plt.show()

In [ ]:
# 'issue_year' wird aus 'issue_d' abgeleitet; es dient der Grafik unten und
# dem Jahresfilter in der nächsten Zelle.
df = add_issue_year(df)

plt.figure(figsize=(12, 7))
sns.countplot(x='issue_year', data=df, hue='issue_year',
              palette='viridis', legend=False)
plt.title('Anzahl vergebener Kredite pro Jahr', fontsize=16)
plt.xlabel('Jahr', fontsize=12)
plt.ylabel('Anzahl Kredite', fontsize=12)
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Einschränkung auf 2015-2018. Die Grafik oben zeigt, dass das Volumen von
# LendingClub über die Jahre um Größenordnungen gewachsen ist und sich die
# Vergabestandards mit ihm verändert haben. Ein aktuelles, homogenes
# Zeitfenster liegt näher an der Population, auf die ein Scoring-Modell
# angewendet würde, als die gesamte Historie 2007-2018.
print(f"Dimensionen des ursprünglichen DataFrames: {df.shape}")

df = filter_issue_years(df, start_year=2015, end_year=2018)

print(f"Dimensionen des gefilterten DataFrames: {df.shape}")
print("\nAnzahl pro Jahr im gefilterten DataFrame:")
print(df['issue_year'].value_counts().sort_index())

In [ ]:
# Nur numerische Spalten für die Korrelation auswählen
numeric_df = df.select_dtypes(include=np.number)

plt.figure(figsize=(12, 10))
sns.heatmap(numeric_df.corr(), cmap='coolwarm')
plt.title('Korrelationsmatrix der numerischen Merkmale')
plt.show()

## 3. Datenbereinigung und Vorverarbeitung

Vier Bereinigungsschritte in dieser Reihenfolge:

1. Nur Kredite mit bekanntem Ausgang behalten und das binäre Zielmerkmal aus
   `loan_status` ableiten.
2. Spalten entfernen, die überwiegend leer sind.
3. Spalten entfernen, die irrelevant sind (Identifikatoren, Freitext) oder den
   Ausgang verraten (Leakage).
4. Zeilen entfernen, die danach noch fehlende Werte enthalten.

Die Reihenfolge ist wichtig: Das Zielmerkmal wird zuerst definiert, damit die
späteren Schritte daran gemessen werden können, und das Entfernen der
Leakage-Spalten läuft nach dem Entfernen der lückenhaften Spalten (Schritt 2),
damit es auch Spalten erfasst, die die 40-%-Schwelle zufällig überstanden
haben.

In [ ]:
# Nur Kredite mit 'Fully Paid' oder 'Charged Off' haben einen endgültigen
# Ausgang. Noch laufende Kredite würden sonst als Nicht-Ausfälle gelabelt,
# obwohl ihr Ausgang schlicht noch nicht bekannt ist.
df = create_target(df)

print("Zielmerkmal 'target' erstellt.")
print(df['target'].value_counts(normalize=True) * 100)

In [ ]:
# Spalten mit mehr als 40 % fehlenden Werten entfernen. Eine überwiegend leere
# Spalte zu imputieren erfindet mehr Daten, als es zurückgewinnt; bei 40 %
# teilen sich die LendingClub-Spalten sauber in "allgemein befüllt" und "nur in
# einem Sonderfall befüllt" (Hardship, Settlement, gemeinsame Anträge).
df = rm_nas(df, threshold=0.4)
print(f"Dimensionen nach Entfernen der Spalten mit >40 % NaNs: {df.shape}")

### Data Leakage: Spalten nach dem Kreditausgang werden entfernt

Die erste Version dieses Notebooks meldete eine ROC AUC von **0,9999**. Auf dem
LendingClub-Datensatz ist ein solcher Wert kein Erfolg, sondern ein bekanntes
Warnsignal für Data Leakage: Mehrere Spalten werden erst befüllt, *nachdem* der
Kredit bereits ausgefallen oder zurückgezahlt ist (erhaltene Zahlungen,
ausstehende Restschuld, Recoveries, der laufend aktualisierte Score
`last_fico_range_*`, Hardship- und Settlement-Felder). Als Features verwendet,
verraten sie dem Modell genau den Ausgang, den es vorhersagen soll.

Die Zelle unten entfernt deshalb zusätzlich zu den irrelevanten Spalten alle
Spalten, die erst nach dem Ausgang bekannt sind. Übrig bleiben die
Informationen, die bei **Kreditvergabe** verfügbar sind, also genau die
Informationen, die ein echtes PD-Scoring-Modell verwenden darf. Die
resultierende AUC ist deutlich niedriger und für ein Ausfallmodell auf dieser
Art von Daten die glaubwürdige (typischerweise im Bereich von etwa 0,65–0,75).

In [ ]:
# Irrelevante erklärende Variablen (Identifikatoren, Freitext, Datumsspalten)
# zusammen mit allen oben beschriebenen Spalten entfernen, die erst nach dem
# Ausgang bekannt sind. Die konkreten Spaltenlisten stehen in
# src/data_processing.py.
df = drop_leakage_columns(df)
print(f"Dimensionen nach Entfernen der irrelevanten und Post-Outcome-Spalten: {df.shape}")

In [ ]:
# Nach dem Entfernen der Spalten betreffen die verbleibenden NaNs nur noch einen
# kleinen Teil der Zeilen; diese Zeilen zu entfernen ist günstiger, als sie zu
# imputieren.
df.dropna(inplace=True)
print(f"Dimensionen nach Entfernen aller verbleibenden NaN-Zeilen: {df.shape}")

## 4. Feature Engineering

Alles, was das Modell sieht, muss numerisch sein. `term` und `emp_length` sind
als Text gespeichert, aber tatsächlich ordinal; sie werden deshalb in Zahlen
umgewandelt statt One-Hot-kodiert. Die übrigen kategorialen Spalten werden
One-Hot-kodiert, mit `drop_first=True`, um die redundante Referenzkategorie zu
vermeiden.

In [ ]:
# engineer_features kopiert seine Eingabe. Wird der unkodierte Frame an einen
# zweiten Namen gebunden, bleibt er für die WOE-Scorecard in Abschnitt 7
# verfügbar, statt hier verworfen zu werden. Es wird keine Kopie erstellt.
df_unencoded = df
df = engineer_features(df_unencoded)

print("Feature Engineering abgeschlossen.")
print(f"Finale Dimensionen der Modellierungsdaten: {df.shape}")

## 5. Modelltraining und Evaluation

Als erstes Modell dient ein Random Forest: Er verarbeitet die Mischung aus
numerischen und One-Hot-kodierten Features ohne weitere Vorverarbeitung, bildet
nichtlineare Zusammenhänge ab und ist robust gegenüber den Ausreißern, von denen
Kreditdaten voll sind. Das interpretierbare Gegenstück, das eine Bank daneben
erwarten würde, eine logistische Regression auf WOE-gebinnten Features, wird in
Abschnitt 7 gebaut und dort mit diesem Modell verglichen.

Ausfälle sind die Minderheitsklasse (rund 20 % der abgeschlossenen Kredite).
Deshalb wird `class_weight="balanced"` statt Resampling verwendet, und der
Train/Test-Split ist stratifiziert, sodass beide Mengen dieselbe Ausfallrate
haben.

In [ ]:
# Der Scaler in split_and_scale wird nur auf dem Trainingsset gefittet, sodass
# keine Information aus dem Testset ins Training gelangt.
X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test, scaler = split_and_scale(
    X, y, test_size=0.3, random_state=42
)

print("Daten in Trainings- und Testset aufgeteilt und Features skaliert.")

In [ ]:
print("Trainiere Random-Forest-Modell... Auf dem vollständigen Datensatz kann das dauern.")

model = train_random_forest(X_train, y_train)
save_model(model, "../04_models/random_forest.joblib")

print("Modelltraining abgeschlossen, Modell gespeichert unter 04_models/random_forest.joblib.")

### Trennschärfe: ROC AUC, Gini und KS

Drei Blickwinkel auf dieselbe Frage: Wie gut trennt der Score die ausgefallenen
Kredite von den zurückgezahlten?

- **ROC AUC** ist die Wahrscheinlichkeit, dass ein zufällig gezogener
  ausgefallener Kredit einen höheren Score erhält als ein zufällig gezogener
  zurückgezahlter. Sie ist schwellenunabhängig und sagt daher nichts über die
  Schwelle von 0,5 aus, die im Classification Report unten verwendet wird.
- **Gini = 2 x AUC - 1** enthält dieselbe Information auf einer Skala, die einen
  Münzwurf auf 0 und ein perfektes Ranking auf 1 abbildet. In dieser Form werden
  Scorecard-Validierung und Modellmonitoring in Banken üblicherweise berichtet;
  deshalb wird er hier ausgewiesen, obwohl er keine Information über die AUC
  hinaus enthält.
- **KS** (Kolmogorov-Smirnov) ist der größte Abstand zwischen den kumulierten
  Score-Verteilungen der beiden Gruppen. Anders als AUC und Gini ist er an einen
  konkreten Punkt auf der Score-Skala gebunden: die Schwelle, an der die Trennung
  am größten ist.

Grobe Orientierung für Retail-PD-Modelle: Ein Gini von etwa 0,30–0,50 (AUC
0,65–0,75) und ein KS von etwa 0,20–0,35 sind der übliche Bereich. Deutlich
höhere Werte auf diesem Datensatz sind eher ein Leakage-Warnsignal als ein
Erfolg; siehe den Abschnitt oben.


In [ ]:
scores = evaluate_model(model, X_test, y_test)

print(f"ROC AUC: {scores['roc_auc']:.4f}")
print(f"Gini:    {scores['gini']:.4f}")
print(f"KS:      {scores['ks']:.4f}")
print("\nClassification Report:")
print(scores['report'])


### Kalibrierung

Die Trennschärfe prüft nur das *Ranking*. Von einem PD-Modell wird zusätzlich
erwartet, dass es auch das *Niveau* trifft: Ein Kredit mit Score 0,10 sollte in
etwa 10 % der Fälle ausfallen. Das wird relevant, sobald der Output in eine
Berechnung des erwarteten Verlusts einfließt. Unter IFRS 9 ist der ECL = PD x
LGD x EAD; eine systematisch zu hohe PD bläht die Risikovorsorge also auf, egal
wie gut das Modell rankt.

Das Reliability-Diagramm unten fasst die vorhergesagten Wahrscheinlichkeiten in
Bins zusammen und trägt die beobachtete Ausfallrate jedes Bins gegen die
mittlere Vorhersage darin ab. Ein perfekt kalibriertes Modell folgt der
Diagonale; Punkte darunter bedeuten, dass das Modell höhere Ausfallraten
vorhersagt, als tatsächlich eintreten, Punkte darüber das Gegenteil.

Das Modell hier wird mit `class_weight="balanced"` trainiert, was ihm bewusst
vorgibt, Ausfälle so zu behandeln, als wären sie genauso häufig wie
Nicht-Ausfälle. Das verbessert das Ranking der Minderheitsklasse, schiebt die
vorhergesagten Wahrscheinlichkeiten aber über die tatsächliche Ausfallrate,
sodass die Kurve erwartungsgemäß unter der Diagonale liegt. Die Lösung ist
nicht, auf die Klassengewichte zu verzichten, sondern die Scores anschließend zu
rekalibrieren (`CalibratedClassifierCV`), bevor sie als Wahrscheinlichkeiten
verwendet werden.


In [ ]:
CalibrationDisplay.from_predictions(
    y_test, scores['y_pred_proba'], n_bins=10, name='Random Forest'
)
plt.title('Kalibrierungskurve (Reliability-Diagramm)')
plt.show()


## 6. Interpretierbarkeit des Modells

Eine Kreditentscheidung muss erklärbar sein: Antragsteller haben Anspruch auf
eine Begründung für eine Ablehnung, und die Modellvalidierung muss prüfen
können, dass das Modell auf ökonomisch sinnvolle Risikotreiber reagiert statt
auf ein Artefakt der Daten. Ein Random Forest liefert eine solche Erklärung
nicht von selbst; deshalb kommen hier zwei Sichten hinzu.

`feature_importances_` beantwortet die Frage „Welche Features nutzt der
Forest?“, SHAP die Frage „In welche Richtung und wie stark verschiebt jedes
Feature *diese* Vorhersage?“. Die zweite Frage ist die, die eine
Kreditrisikofunktion tatsächlich stellt.

In [ ]:
importances = pd.Series(model.feature_importances_, index=X_train.columns)

plt.figure(figsize=(10, 7))
importances.nlargest(15).sort_values().plot.barh()
plt.title('Top 15 Features nach Random-Forest-Importance')
plt.xlabel('Mittlere Abnahme der Impurity')
plt.tight_layout()
plt.show()

Das Balkendiagramm oben ordnet die Features nach der mittleren Abnahme der
Impurity. Dieses Maß hat zwei bekannte Grenzen: Es sagt nichts über die
*Richtung* eines Effekts, und es bevorzugt systematisch stetige Features und
solche mit vielen Ausprägungen gegenüber binären, weil diese den Bäumen mehr
mögliche Split-Punkte bieten. Eine One-Hot-Spalte kann hier daher unwichtig
aussehen und trotzdem für einzelne Entscheidungen relevant sein.

SHAP-Werte beheben beide Probleme. Sie ordnen jede einzelne Vorhersage ihren
Features zu, tragen also ein Vorzeichen, und sind über Feature-Typen hinweg
vergleichbar. Wie im Issue vorgesehen, werden sie auf einer Zufallsstichprobe
statt auf dem gesamten Testset berechnet: `TreeExplainer` ist auf einem Forest
mit so vielen Zeilen und Spalten langsam, und einige tausend Zeilen reichen für
einen stabilen Summary-Plot.

In [ ]:
X_shap = X_test.sample(min(3000, len(X_test)), random_state=42)

# Index 1 der letzten Achse wählt die Beiträge zur Ausfallklasse aus.
shap_values = shap.TreeExplainer(model)(X_shap)[:, :, 1]

shap.plots.beeswarm(shap_values, max_display=15, show=False)
plt.title('SHAP-Übersicht: Einfluss auf die vorhergesagte Ausfallwahrscheinlichkeit')
plt.tight_layout()
plt.show()

### Den Summary-Plot lesen

Jeder Punkt ist ein Kredit. Seine Position auf der x-Achse ist der Beitrag
dieses Features zur Ausfallwahrscheinlichkeit dieses Kredits: Rechts von null
schiebt er die Vorhersage Richtung Ausfall, links von null davon weg. Die Farbe
ist der Feature-Wert; „rot rechts“ heißt also, dass hohe Werte des Features das
vorhergesagte Risiko erhöhen.

Eine Einschränkung, die speziell dieses Notebook betrifft: Die Features wurden
vor dem Training standardisiert, die Farbskala bezieht sich also auf die
skalierten Werte. Die Standardisierung ist monoton, rot bedeutet daher weiterhin
einen hohen Wert in der ursprünglichen Einheit, aber die Zahlen auf dem
Farbbalken sind keine Dollar, Prozent oder Jahre.

Ob das Muster ökonomisch Sinn ergibt, ist der eigentliche Test, und drei
Features verdienen eine explizite Prüfung:

- **`int_rate`** dürfte der stärkste einzelne Treiber sein, wobei hohe Zinsen
  die Vorhersage Richtung Ausfall schieben. Das ist kein Leakage, denn der Zins
  wird bei Vergabe festgelegt, verdient aber eher einen Vorbehalt als Applaus:
  LendingClub leitet den Zins aus seiner eigenen internen Bonitätsstufe ab, und
  `grade`/`sub_grade` wurden als Identifikatoren aus dem Feature-Set entfernt.
  Ein Modell, das von `int_rate` dominiert wird, reproduziert daher größtenteils
  die Underwriting-Entscheidung der Plattform, statt den Kreditnehmer
  unabhängig zu beurteilen. Eine Bank, die eine Scorecard baut, würde prüfen,
  wie viel Trennschärfe ohne dieses Feature übrig bleibt.
- **`dti`** (Debt-to-Income, Verhältnis von Schuldendienst zu Einkommen) ist
  ökonomisch der am besten begründbare Treiber: Er misst, wie viel des
  Einkommens bereits für den Schuldendienst gebunden ist, und ein Kreditnehmer
  mit wenig Spielraum kann einen Einkommensschock schlechter abfedern. Ein
  höherer `dti` sollte Richtung Ausfall schieben; wäre das nicht so, wäre das
  Modell verdächtig, nicht die Theorie.
- **`term`** sollte Kredite mit 60 Monaten Laufzeit als riskanter zeigen als
  solche mit 36 Monaten, aus zwei sich verstärkenden Gründen: Der Kredit ist
  länger ungünstigen Ereignissen ausgesetzt, und Kreditnehmer, die die längere
  Laufzeit wählen, um eine niedrigere Rate zu bekommen, sind tendenziell schon
  von vornherein die finanziell angespannteren. Das ist ein gutes Beispiel für
  ein Feature, dessen Effekt teils Laufzeit und teils Selbstselektion ist.

In [ ]:
riskiest = int(model.predict_proba(X_shap)[:, 1].argmax())

shap.plots.waterfall(shap_values[riskiest], max_display=12, show=False)
plt.title('Einzelfall: Warum dieser Kredit als hochriskant eingestuft wird')
plt.tight_layout()
plt.show()

Der Waterfall-Plot erklärt einen einzelnen Kredit, nämlich den, den das Modell
in der Stichprobe als den riskantesten einstuft. Er beginnt beim Basiswert des
Modells (`E[f(x)]`) und addiert den Beitrag jedes Features, bis er die
Vorhersage für diesen Kredit erreicht (`f(x)`).

`E[f(x)]` ist nicht die durchschnittliche Vorhersage über diese Stichprobe.
`TreeExplainer` leitet ihn aus den Gewichten der Trainingsdaten ab, die in den
Bäumen gespeichert sind, und diese Gewichte tragen das
`class_weight="balanced"`, das `src/train.py` setzt. Der Basiswert liegt
deshalb bei etwa 0,5 statt bei der tatsächlichen Ausfallrate des Portfolios.
Jeder Beitrag im Plot wird gegen diesen balancierten Ausgangspunkt gemessen; das
ist derselbe Vorbehalt wie im Abschnitt zur Kalibrierung: Dieses Modell rankt,
es liefert keine PDs.

Genau dieses Format braucht ein Ablehnungsbescheid (Adverse Action Notice) oder
ein Kreditsachbearbeiter, der eine Entscheidung übersteuert: nicht „das Modell
sagte 0,87“, sondern welche Merkmale genau dieses Antrags dorthin geführt haben
und wie stark.


### Einordnung der Ergebnisse

Das Modell oben verwendet nur Features, die bei Kreditvergabe bekannt sind. Der
korrigierte Lauf auf dem vollständigen Datensatz steht noch aus (die Roh-CSV ist
nicht Teil dieses Repositorys), deshalb sind die konkreten Metriken hier noch
nicht eingetragen.

Das korrigierte Modell muss eine AUC weit unter den 0,9999 der ersten Version
zeigen, und genau dieser Rückgang ist der Punkt: Der frühere Wert kam von
Spalten nach dem Kreditausgang, die das Zielmerkmal verraten haben (siehe den
Abschnitt zu Data Leakage oben), nicht von Vorhersagekraft. Ein Wert im Bereich
von 0,65–0,75 ist das, was ein realistisches PD-Modell auf LendingClub-Daten
erreicht, und die Zahl, die sich in einem Kreditrisiko-Kontext vertreten lässt;
in der Form, in der eine Scorecard-Validierung ihn berichten würde, entspricht
das einem Gini von etwa 0,30–0,50.

Für die fachliche Interpretation ist der Recall der Klasse 'Charged Off (1)' die
relevante Größe: Er zeigt, wie viele der tatsächlich ausfallenden Kredite das
Modell markiert. Precision und Recall stehen nun deutlich im Zielkonflikt; die
Entscheidungsschwelle muss daher nach den Kosten eines übersehenen Ausfalls
gegenüber einem abgelehnten guten Kunden gewählt werden. Der KS-Wert oben gibt
an, wie groß diese Trennung wird, nicht bei welchem Score sie auftritt; er
begrenzt also, was eine Schwelle erreichen kann, statt eine zu benennen.

Die Kalibrierungskurve ist davon getrennt zu lesen: Sie sagt nichts über die
Ranking-Qualität aus, nur darüber, ob die vorhergesagten Wahrscheinlichkeiten in
einer Berechnung des erwarteten Verlusts als PDs verwendet werden können. Mit
`class_weight="balanced"` können sie das ohne Rekalibrierung nicht. Für ein
Ranking-Modell ist das akzeptabel, muss aber ausdrücklich gesagt werden, statt
stillschweigend vorausgesetzt zu werden.

## 7. Benchmark: Logistische Regression als Scorecard (WOE / IV)

Der Random Forest ist das moderne Modell. Das Modell, das eine Bank
üblicherweise vor einer Aufsicht verteidigen muss, ist das ältere: eine
**Scorecard**, also eine logistische Regression auf
Weight-of-Evidence-transformierten Features. Durch sie werden die vorherigen
Abschnitte zu einem Vergleich statt zu einem Einzelergebnis.

**Weight of Evidence (WOE)** teilt ein Feature in Bins ein und ersetzt jeden
Wert durch die Log-Odds seines Bins:

`WOE = ln(Anteil der Nicht-Ausfälle im Bin / Anteil der Ausfälle im Bin)`

Ein Bin mit relativ vielen zurückgezahlten Krediten erhält einen positiven WOE,
einer mit relativ vielen Ausfällen einen negativen. Diese Transformation macht
den Ansatz für eine Modellvalidierung akzeptabel:

- Der Zusammenhang zwischen Feature und Zielmerkmal muss nicht mehr linear sein:
  Die Bins fangen die Form auf, und die Regression bleibt linear in den
  WOE-Werten.
- Ausreißer können keinen Koeffizienten verschieben; sie fallen in einen
  Randbin. Kreditdaten sind voll davon (insbesondere `annual_inc`).
- Fehlende Werte werden nicht imputiert, sondern erhalten einen eigenen Bin,
  sodass „nicht angegeben“ zu einer expliziten Risikoaussage wird statt zu einer
  stillen Schätzung. Dieser Bin bleibt hier leer, weil Abschnitt 3 die
  verbleibenden NaN-Zeilen bereits entfernt hat; in einem produktiven Modell ist
  er einer der Hauptgründe, überhaupt zu binnen.
- Jeder Score ist nachvollziehbar: Der Antragsteller ist in diesem Bin gelandet,
  der so viele Punkte wert ist. Genau das braucht ein Ablehnungsbescheid.

**Information Value (IV)** fasst den WOE eines Features zu einer Zahl zusammen,
die angibt, wie gut es die beiden Klassen trennt. Die übliche Lesart:

| IV | Lesart |
| --- | --- |
| < 0,02 | kaum prädiktiv – entfernen |
| 0,02 – 0,1 | schwach |
| 0,1 – 0,3 | mittel |
| 0,3 – 0,5 | stark |
| > 0,5 | verdächtig stark – erst auf Leakage prüfen, dann glauben |

Die letzte Zeile folgt demselben Instinkt wie Abschnitt 3: Ein Feature, das
Ausfälle fast perfekt trennt, ist meist ein Feature, das den Ausgang bereits
kennt.

Das Binning selbst übernimmt `optbinning`, das es als Optimierungsproblem löst
(IV maximieren unter einer Mindestgröße pro Bin und Monotonie), statt an festen
Quantilen zu schneiden.

In [ ]:
# Die Scorecard bint die *unkodierten* Features. Die WOE-Gruppierung arbeitet
# auf den Rohkategorien, die One-Hot-Kodierung aus Abschnitt 4 würde sie nur
# wieder auseinanderreißen; aus demselben Grund entfällt die ordinale
# Umwandlung von 'term' und 'emp_length', da das Binning deren Ausprägungen
# selbst nach Ausfallrate gruppiert.
X_woe = df_unencoded.drop('target', axis=1)
categorical_variables = list(X_woe.select_dtypes(include=['str', 'object']).columns)

# split_and_scale behält den ursprünglichen Index; die Auswahl darüber legt also
# genau dieselben Kredite in dieselbe Menge für beide Modelle, und die Metriken
# bleiben vergleichbar.
X_woe_train = X_woe.loc[X_train.index]
X_woe_test = X_woe.loc[X_test.index]

scorecard = train_scorecard(X_woe_train, y_train, categorical_variables)

n_numeric = X_woe.shape[1] - len(categorical_variables)
print(f"Scorecard trainiert auf {len(categorical_variables)} kategorialen und "
      f"{n_numeric} numerischen Kandidaten-Features.")

### Information Value pro Feature

Die Tabelle unten macht die Feature-Auswahl explizit. `selected` ist `False` für
jedes Feature, dessen IV unter 0,02 geblieben ist. Diese Features werden
entfernt, bevor die Regression sie sieht; so hält sich eine Scorecard
klassischerweise klein genug, um von Hand geprüft zu werden.

In [ ]:
binning = scorecard.named_steps['woe']
iv_table = binning.summary()[['name', 'dtype', 'n_bins', 'iv', 'selected']]
iv_table = iv_table.sort_values('iv', ascending=False)

print(iv_table.to_string(index=False))

dropped = iv_table.loc[~iv_table['selected'], 'name'].tolist()
suspicious = iv_table.loc[iv_table['iv'] > 0.5, 'name'].tolist()

print(f"\nEntfernt, IV unter 0,02: {dropped}")
print(f"Leakage-Kandidaten, IV über 0,5: {suspicious}")

Alles, was als Leakage-Kandidat ausgegeben wird, ist ein Anlass zum Hinsehen,
kein Urteil. Drei Punkte sind in dieser Tabelle besonders zu prüfen:

- **`int_rate`** ist das Feature, das ganz oben zu erwarten ist. LendingClub
  legt den Zins anhand seiner eigenen Bonitätsstufe fest; ein hoher IV ist hier
  also die eigene Kreditbeurteilung der Plattform, die durchscheint, und keine
  Post-Outcome-Spalte, die Abschnitt 3 übersehen hat. Der Zins ist bei Vergabe
  bekannt und bleibt daher im Modell. Ein Modell, das sich darauf stützt,
  reproduziert aber teilweise die Scorecard eines anderen, und eine
  Modelldokumentation muss das sagen.
- **`fico_range_low` und `fico_range_high`** sind die beiden Enden eines vier
  Punkte breiten Bands, tragen also fast dieselbe Information und werden fast
  denselben IV zeigen. Der IV wird pro Feature berechnet und kann das nicht
  erkennen; ein produktives Modell würde eines der beiden behalten und das
  andere entfernen, statt der Regression zwei Kopien derselben Variable zu
  geben.
- **`issue_year`** ist die offene Frage aus der Leakage-Prüfung: Es ist bei
  Vergabe bekannt und damit kein Leakage, lässt das Modell als Zeitindex aber
  den Jahrgang (Vintage) statt des Risikos lernen. In der IV-Spalte wird das
  anhand von Evidenz entschieden: Liegt der Wert auf dem vollständigen Datensatz
  über der Schwelle von 0,02, trägt der Jahrgang Signal, und das Feature
  verdient eine bewusste Entscheidung statt einer stillschweigenden.

In [ ]:
scorecard_scores = evaluate_model(scorecard, X_woe_test, y_test)

comparison = pd.DataFrame(
    {
        'Random Forest': [scores['roc_auc'], scores['gini'], scores['ks']],
        'LR Scorecard (WOE)': [
            scorecard_scores['roc_auc'],
            scorecard_scores['gini'],
            scorecard_scores['ks'],
        ],
    },
    index=['ROC AUC', 'Gini', 'KS'],
)

print(comparison.round(4).to_string())
print("\nClassification Report der Scorecard:")
print(scorecard_scores['report'])

### Welches Modell in der Praxis eingesetzt würde

Beide Modelle werden auf denselben Testzeilen mit denselben drei Metriken
bewertet, die Zahlen oben sind also direkt vergleichbar. Jedes der beiden kann
vorne liegen, und genau das ist der aufschlussreiche Teil.

**Den Abstand lesen.** Der Forest kann Interaktionen zwischen Features nutzen,
die die Scorecard als in den WOE-Werten additives Modell überhaupt nicht
abbilden kann. Ein klarer Vorsprung des Forests bedeutet also, dass es
nichtlineare Struktur gibt, die der Scorecard entgeht. Die übliche Reaktion ist
nicht, die Scorecard aufzugeben, sondern anders zu binnen oder die Interaktion
als eigenes Feature aufzunehmen. Eine Scorecard, die mit dem Forest gleichzieht
oder ihn schlägt, sagt das Gegenteil: Die Zusammenhänge in diesen Daten sind
nah genug an monoton, dass Flexibilität nichts bringt, und das einfachere Modell
ist schlicht das bessere.

**Wann die Scorecard aus anderen Gründen als der Metrik gewinnt.** Ein
regulatorisches PD-Modell, also ein IRB-Ratingsystem unter Basel oder die PD,
die in eine IFRS-9-Berechnung des erwarteten Verlusts einfließt, muss über Jahre
erklärt, validiert und verteidigt werden. Die Scorecard ist additiv und
binweise; jede Entscheidung zerfällt also in eine Handvoll Aussagen der Form
„dieser Antragsteller liegt in diesem Bin, der so viele Punkte wert ist“.
Abgelehnte Antragsteller können eine Begründung erhalten, eine
Validierungseinheit kann jeden Bin gegen ihre eigene Erwartung zur
Risikorichtung prüfen, und das Monitoring kann beobachten, wie sich die
Bin-Besetzungen über die Zeit verschieben. Die Koeffizienten sind wenige und
stabil, und das ganze Modell passt auf eine Seite.

**Wann der Forest gewinnt.** Wo der Output ein intern genutztes Ranking ist und
die Kosten einer Fehlentscheidung begrenzt sind (Portfolio-Triage,
Priorisierung im Inkasso, eine Frühwarnliste oder ein Challenger-Modell, das
prüft, ob die Scorecard Signal liegen lässt), ist zusätzliche Trennschärfe mehr
wert als Erklärbarkeit, zumal die SHAP-Werte aus Abschnitt 6 einen guten Teil
davon zurückholen.

**Warum das nicht einfach Genauigkeit gegen Interpretierbarkeit ist.** Zwei
Eigenschaften der Scorecard betreffen Robustheit, nicht Transparenz: Das
Binning begrenzt den Einfluss von Ausreißern, und jedes Feature hat einen
expliziten Bin für fehlende Werte. Der Forest leistet beides konstruktionsbedingt
nicht, und beides wiegt auf Kreditdaten meist schwerer als der
Metrikunterschied.

Die ehrliche Zusammenfassung: Eine Bank würde beide bauen. Die Scorecard ist das
maßgebliche Modell (Model of Record), der Forest der Challenger, der sie ehrlich
hält.